In [1]:
!pip install --quiet gradio python-dotenv openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Same pattern that just worked for Black Tongue
load_dotenv()

for k in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "DEEPSEEK_API_KEY"]:
    v = os.getenv(k, "")
    print(f"{'✅' if v else '❌'} {k}: {v[:8]}…{v[-4:]}" if v else f"❌ {k}: not set")

✅ OPENROUTER_API_KEY: sk-or-v1…0062
✅ GROQ_API_KEY: gsk_Kr1e…pbx1
✅ DEEPSEEK_API_KEY: sk-f9d15…85ae


In [3]:
PROVIDERS = {
    "openrouter": {
        "api_key": os.getenv("OPENROUTER_API_KEY"),
        "base_url": "https://openrouter.ai/api/v1",
        "model": os.getenv("OPENROUTER_MODEL", "meta-llama/llama-3.1-8b-instruct"),
    },
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",
        "model": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
    },
    "deepseek": {
        "api_key": os.getenv("DEEPSEEK_API_KEY"),
        "base_url": "https://api.deepseek.com/v1",
        "model": os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
    },
}

PROVIDER_ORDER = ["openrouter", "groq", "deepseek"]

for name, cfg in PROVIDERS.items():
    print(f"{name:12s} → {'✅ loaded' if cfg['api_key'] else '❌ MISSING'}")

openrouter   → ✅ loaded
groq         → ✅ loaded
deepseek     → ✅ loaded


In [4]:
import gradio as gr, inspect

GRADIO_VERSION = gr.__version__
print(f"🔎 Gradio version: {GRADIO_VERSION}")

_chatbot_sig = inspect.signature(gr.Chatbot.__init__).parameters
_blocks_sig  = inspect.signature(gr.Blocks.__init__).parameters
_launch_sig  = inspect.signature(gr.Blocks.launch).parameters

HAS_TYPE_PARAM        = "type" in _chatbot_sig
HAS_BUBBLE_FULL_WIDTH = "bubble_full_width" in _chatbot_sig
CSS_IN_BLOCKS         = "css" in _blocks_sig
CSS_IN_LAUNCH         = "css" in _launch_sig

HISTORY_FORMAT = "messages" if int(GRADIO_VERSION.split(".")[0]) >= 5 else "tuples"
print(f"👉 HISTORY_FORMAT = {HISTORY_FORMAT}")

🔎 Gradio version: 6.27.0
👉 HISTORY_FORMAT = messages


In [5]:
LANGUAGES = {
    "English":              "English",
    "Nigerian Pidgin":      "Nigerian Pidgin English",
    "Hausa":                "Hausa",
    "Yoruba":               "Yorùbá",
    "Igbo":                 "Igbo",
    "French":               "French",
    "Spanish":              "Spanish",
    "Portuguese":           "Portuguese",
    "German":               "German",
    "Arabic":               "Arabic",
    "Chinese (Simplified)": "Simplified Chinese",
    "Hindi":                "Hindi",
    "Swahili":              "Swahili",
    "Russian":              "Russian",
    "Japanese":             "Japanese",
}
print(f"{len(LANGUAGES)} languages loaded.")

15 languages loaded.


In [6]:
def build_system_prompt(language: str, level: str, company: str = "BeGinQode") -> str:
    return (
        f"You are {company}, a friendly, patient coding tutor who helps absolute "
        f"beginners understand, debug, and write code.\n\n"
        f"REPLY LANGUAGE: {language}\n"
        f"LEARNER LEVEL: {level}\n\n"
        f"YOUR JOB:\n"
        f"- Explain code line-by-line in plain language (no jargon unless you define it).\n"
        f"- Debug errors by (1) quoting the exact error, (2) explaining WHY it happens, "
        f"(3) showing the fix, (4) explaining the fix.\n"
        f"- When asked to write code, provide complete, runnable examples with comments.\n"
        f"- Use analogies to everyday things when helpful.\n\n"
        f"STRICT RULES:\n"
        f"1. Always reply in {language}. Even if the user writes in another language, "
        f"respond in {language} unless they explicitly ask you to switch.\n"
        f"2. Always wrap code in fenced code blocks with the language tag, e.g. ```python.\n"
        f"3. After every code block, add a short 'What this does' summary in plain words.\n"
        f"4. Anticipate common beginner mistakes and warn about them.\n"
        f"5. Never assume prior knowledge. If unsure of the user's level, ask.\n"
        f"6. If the question is ambiguous, ask ONE clarifying question before answering.\n"
    )


def call_llm(messages, temperature: float, max_tokens: int):
    last_error = None
    for name in PROVIDER_ORDER:
        cfg = PROVIDERS[name]
        if not cfg["api_key"]:
            last_error = f"{name}: missing API key"
            continue
        try:
            client = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"])
            resp = client.chat.completions.create(
                model=cfg["model"],
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content, f"✅ {name} ({cfg['model']})"
        except Exception as e:
            last_error = f"{name}: {e}"
            continue
    return (
        "⚠️ Sorry, all language models are currently unavailable.\n\n"
        f"(Details: {last_error})",
        "❌ all providers failed",
    )

print("✅ Prompt + LLM caller ready.")

✅ Prompt + LLM caller ready.


In [7]:
CUSTOM_CSS = """
:root {
    --bq-bg: #0d0b14;
    --bq-surface: #16121f;
    --bq-purple: #8a4fff;
    --bq-purple-soft: #a175ff;
    --bq-text: #e9e4f5;
    --bq-muted: #9a8fb8;
}
.gradio-container {
    background: var(--bq-bg) !important;
    color: var(--bq-text) !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
    max-width: 1000px !important;
    margin: auto !important;
}
#bq-header { text-align:center; padding:20px 0 8px 0; border-bottom:1px solid rgba(138,79,255,0.25); margin-bottom:16px; }
#bq-header h1 { color: var(--bq-purple-soft); letter-spacing:2px; font-weight:700; font-size:2rem; margin:0; text-shadow:0 0 18px rgba(138,79,255,0.55); }
#bq-header p { color: var(--bq-muted); margin:4px 0 0 0; font-size:0.95rem; }
#bq-chatbot { background: var(--bq-surface) !important; border:1px solid rgba(138,79,255,0.3) !important; border-radius:14px !important; }
#bq-chatbot .message.user { background: linear-gradient(135deg,#5a2ecc,#8a4fff) !important; color:#fff !important; border-radius:14px 14px 4px 14px !important; }
#bq-chatbot .message.bot  { background:#1e1830 !important; color: var(--bq-text) !important; border:1px solid rgba(138,79,255,0.25) !important; border-radius:14px 14px 14px 4px !important; }
/* Code blocks */
#bq-chatbot pre, #bq-chatbot code {
    background: #0a0812 !important;
    color: #d6c9ff !important;
    border-radius: 8px !important;
    font-family: 'JetBrains Mono', 'Consolas', monospace !important;
}
#bq-chatbot pre { padding: 10px !important; border: 1px solid rgba(138,79,255,0.35) !important; }
button.primary, .gr-button-primary { background: linear-gradient(135deg,#6a35d6,#8a4fff) !important; border:none !important; color:#fff !important; font-weight:600 !important; }
button.primary:hover { background:#9a63ff !important; }
input, textarea, .gr-dropdown, select { background: var(--bq-surface) !important; color: var(--bq-text) !important; border:1px solid rgba(138,79,255,0.35) !important; border-radius:10px !important; }
footer { display: none !important; }
"""
print("✅ CSS ready.")

✅ CSS ready.


In [8]:
TEMPERATURE = 0.3   # lower = more factual for code
MAX_TOKENS  = 2048  # higher ceiling for code snippets

def respond(user_message, chat_history, language_label, level_label):
    # --- Error handling: empty input ---
    if not user_message or not user_message.strip():
        return chat_history, ""

    # --- Error handling: message too long ---
    if len(user_message) > 8000:
        err = "⚠️ Your message is too long. Please split it into smaller parts (max ~8000 characters)."
        if HISTORY_FORMAT == "messages":
            chat_history = chat_history + [
                {"role": "user", "content": user_message[:200] + "…"},
                {"role": "assistant", "content": err},
            ]
        else:
            chat_history = chat_history + [[user_message[:200] + "…", err]]
        return chat_history, ""

    try:
        language = LANGUAGES.get(language_label, "English")
        system_prompt = build_system_prompt(language, level_label)

        messages = [{"role": "system", "content": system_prompt}]
        for turn in chat_history:
            if isinstance(turn, dict) and "role" in turn and "content" in turn:
                messages.append({"role": turn["role"], "content": turn["content"]})
            elif isinstance(turn, (list, tuple)) and len(turn) == 2:
                u, b = turn
                if u: messages.append({"role": "user", "content": u})
                if b: messages.append({"role": "assistant", "content": b})
        messages.append({"role": "user", "content": user_message})

        reply, status = call_llm(messages, TEMPERATURE, MAX_TOKENS)
        reply = f"{reply}\n\n_— {status}_"

    except Exception as e:
        # --- Global error handling: never crash the UI ---
        reply = (
            f"⚠️ Something went wrong while processing your request.\n\n"
            f"**Error type:** `{type(e).__name__}`\n"
            f"**Message:** `{str(e)[:300]}`\n\n"
            f"Try rephrasing your question, or click **Reset** to start fresh."
        )

    if HISTORY_FORMAT == "messages":
        chat_history = chat_history + [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": reply},
        ]
    else:
        chat_history = chat_history + [[user_message, reply]]

    return chat_history, ""


def reset_chat():
    return [], ""

print(f"✅ Handler ready (temp={TEMPERATURE}, max_tokens={MAX_TOKENS}).")

✅ Handler ready (temp=0.3, max_tokens=2048).


In [9]:
_blocks_kwargs = {"title": "BeGinQode — Understand code better with AI"}
if CSS_IN_BLOCKS:
    _blocks_kwargs["css"] = CUSTOM_CSS

with gr.Blocks(**_blocks_kwargs) as demo:
    gr.HTML("""
        <div id="bq-header">
            <h1>BeGinQode</h1>
            <p>Understand better code with AI</p>
        </div>
    """)

    with gr.Row():
        language_dd = gr.Dropdown(
            choices=list(LANGUAGES.keys()),
            value="English",
            label="🌍 Reply Language",
            scale=2,
        )
        level_dd = gr.Dropdown(
            choices=[
                "Absolute beginner (no coding experience)",
                "Beginner (knows basics of one language)",
                "Intermediate (comfortable with functions & loops)",
            ],
            value="Absolute beginner (no coding experience)",
            label="🎓 My level",
            scale=2,
        )

    _chat_kwargs = {"elem_id": "bq-chatbot", "height": 520, "label": "BeGinQode Chat"}
    if HAS_TYPE_PARAM:
        _chat_kwargs["type"] = HISTORY_FORMAT
    elif HAS_BUBBLE_FULL_WIDTH:
        _chat_kwargs["bubble_full_width"] = False

    chatbot = gr.Chatbot(**_chat_kwargs)

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Paste code, describe a bug, or ask how something works…",
            label="",
            scale=8,
            lines=2,
            container=False,
        )
        send = gr.Button("Send", variant="primary", scale=1)
        clear = gr.Button("Reset", scale=1)

    # Example prompts for beginners
    gr.Examples(
        examples=[
            ["Explain this Python code:\n\ndef add(a, b):\n    return a + b\n\nprint(add(2,3))"],
            ["Why do I get this error?\n\nTypeError: 'int' object is not subscriptable"],
            ["Write a function that reverses a string in JavaScript."],
            ["What is a variable? Explain like I'm 5."],
        ],
        inputs=msg,
        label="💡 Try one of these",
    )

    send.click(respond, [msg, chatbot, language_dd, level_dd], [chatbot, msg])
    msg.submit(respond, [msg, chatbot, language_dd, level_dd], [chatbot, msg])
    clear.click(reset_chat, None, [chatbot, msg])

print(f"✅ UI built (format={HISTORY_FORMAT}).")

✅ UI built (format=messages).


In [10]:
try:
    demo.close()
except Exception:
    pass

import inspect as _inspect
_launch_params = set(_inspect.signature(gr.Blocks.launch).parameters.keys())

_candidates = {
    "server_name": "0.0.0.0",
    "server_port": 7861,      # different port so it doesn't clash with Black Tongue
    "share":       True,
    "css":         CUSTOM_CSS,
}
_launch_kwargs = {k: v for k, v in _candidates.items() if k in _launch_params}

print(f"🚀 Launching with: {list(_launch_kwargs.keys())}")
demo.launch(**_launch_kwargs)

🚀 Launching with: ['server_name', 'server_port', 'share', 'css']
* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://8bc40196f4064af116.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
